# DOJ: Hiring A Lot of Attorneys — But Still Shrinking

**Repo:** [github.com/abigailhaddad/doj-lawyer-hiring](https://github.com/abigailhaddad/doj-lawyer-hiring)

Tracks headcount, actual new hires, departures, and job postings for DOJ series 0905 (General Attorney) — before and after January 20, 2025.

## Data sources

**OPM EHRI** — [`impactproject/opm-ehri-data`](https://huggingface.co/datasets/impactproject/opm-ehri-data) on HuggingFace  
Two datasets within EHRI: monthly accessions (new hires) and monthly employment snapshots (headcount). Both queried server-side via DuckDB `hf://` — no local download.

**USAJobs** — R2 parquet mirror at `pub-317c58882ec04f329b63842c1eb65b0c.r2.dev`  
Historical postings from [`abigailhaddad/usajobs_historical`](https://github.com/abigailhaddad/usajobs_historical), stored as annual parquet files. Queried via DuckDB — no download.

Both sources use OPM's four-digit occupational series codes. Series `0905` is General Attorney. Filtering both to DOJ + 0905 puts them on the same population.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

INAUGURATION = '2025-01-20'
CHART_START  = '2024-01-01'
R2 = 'https://pub-317c58882ec04f329b63842c1eb65b0c.r2.dev'
HF = 'hf://datasets/impactproject/opm-ehri-data'

conn = duckdb.connect()

## Helper: pick latest version per month

EHRI files exist in multiple versions (v1, v2, v3) as OPM revises data. We pick the highest version per month.

In [ ]:
def latest_version_urls(glob_pattern, year_min=2022):
    all_files = conn.execute(f"SELECT file FROM glob('{glob_pattern}')").df()['file'].tolist()
    best = {}
    for f in all_files:
        name = f.split('/')[-1]
        parts = name.replace('.parquet', '').split('_')
        yyyymm, ver = parts[1], int(parts[2][1:])
        if int(yyyymm[:4]) >= year_min:
            if yyyymm not in best or ver > best[yyyymm][1]:
                best[yyyymm] = (f, ver)
    return [v[0] for v in sorted(best.values(), key=lambda x: x[0])]

## Part 1: EHRI Accessions — Who Actually Got Hired

In [ ]:
acc_urls = latest_version_urls(f'{HF}/accessions/*.parquet')
url_list = ', '.join(f"'{u}'" for u in acc_urls)

acc_df = conn.execute(f"""
    SELECT 
        regexp_extract(filename, 'accessions_([0-9]{{6}})', 1) AS yyyymm,
        SUM(CAST(count AS INTEGER)) AS accessions
    FROM read_parquet([{url_list}], filename=true)
    WHERE agency ILIKE '%JUSTICE%'
      AND occupational_series_code = '0905'
    GROUP BY 1 ORDER BY 1
""").df()

acc_df['date'] = pd.to_datetime(acc_df['yyyymm'], format='%Y%m')
print(f"Accessions: {acc_df['date'].min().strftime('%Y-%m')} → {acc_df['date'].max().strftime('%Y-%m')}")
acc_df.tail(8)

## Part 2: EHRI Employment — Total Headcount

In [ ]:
emp_urls = latest_version_urls(f'{HF}/employment/*.parquet')
emp_url_list = ', '.join(f"'{u}'" for u in emp_urls)

emp_df = conn.execute(f"""
    SELECT 
        regexp_extract(filename, 'employment_([0-9]{{6}})', 1) AS yyyymm,
        SUM(CAST(count AS INTEGER)) AS headcount
    FROM read_parquet([{emp_url_list}], filename=true)
    WHERE agency = 'DEPARTMENT OF JUSTICE'
      AND occupational_series_code = '0905'
    GROUP BY 1 ORDER BY 1
""").df()

emp_df['date'] = pd.to_datetime(emp_df['yyyymm'], format='%Y%m')
emp_df['net_change'] = emp_df['headcount'].diff()

peak   = emp_df.loc[emp_df['headcount'].idxmax()]
latest = emp_df.iloc[-1]
loss   = int(peak.headcount - latest.headcount)
print(f"Peak:   {int(peak.headcount):,} ({peak.date.strftime('%b %Y')})")
print(f"Latest: {int(latest.headcount):,} ({latest.date.strftime('%b %Y')})")
print(f"Net loss: {loss:,} attorneys ({loss/peak.headcount*100:.0f}%)")
emp_df.tail(8)

## Part 3: USAJobs Postings

The `occupationalSeries` field can contain multiple codes (e.g. `0905 - Attorney; 0950 - Paralegal`), so we filter with `LIKE '%0905%'`.

In [ ]:
years     = list(range(2019, 2027))
hist_urls = [f"'{R2}/data/historical_jobs_{y}.parquet'" for y in years]
curr_urls = [f"'{R2}/data/current_jobs_{y}.parquet'"   for y in [2024, 2025, 2026]]

# Both historical and current parquets use positionOpenDate and JobCategories.
# The same job can appear in both parquets; naively unioning them double-counts
# jobs where hiringAgencyName differs (current API sometimes returns department
# name instead of bureau name when OrganizationName is null). Deduplicate by
# control number, preferring the record with the more specific agency name,
# then historical over current as a tiebreak.
post_df = conn.execute(f"""
    WITH combined AS (
        SELECT usajobsControlNumber,
               positionOpenDate AS open_date,
               CASE WHEN lower(hiringAgencyName) != lower(hiringDepartmentName) THEN 0 ELSE 1 END AS _dept_fallback,
               0 AS _src
        FROM read_parquet([{', '.join(hist_urls)}])
        WHERE hiringDepartmentName = 'Department of Justice'
          AND JobCategories LIKE '%0905%'
          AND positionOpenDate IS NOT NULL
        UNION ALL
        SELECT usajobsControlNumber,
               positionOpenDate AS open_date,
               CASE WHEN lower(hiringAgencyName) != lower(hiringDepartmentName) THEN 0 ELSE 1 END AS _dept_fallback,
               1 AS _src
        FROM read_parquet([{', '.join(curr_urls)}], union_by_name=true)
        WHERE hiringDepartmentName = 'Department of Justice'
          AND JobCategories LIKE '%0905%'
          AND positionOpenDate IS NOT NULL
    ),
    deduped AS (
        SELECT * FROM combined
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY usajobsControlNumber
            ORDER BY _dept_fallback, _src
        ) = 1
    )
    SELECT LEFT(open_date, 7) AS month,
           COUNT(DISTINCT usajobsControlNumber) AS postings
    FROM deduped
    GROUP BY 1 ORDER BY 1
""").df()

post_df['date'] = pd.to_datetime(post_df['month'] + '-01')
post_df = post_df[post_df['date'] >= '2022-01-01'].reset_index(drop=True)
print(f"Postings: {post_df['month'].min()} → {post_df['month'].max()}")
post_df.tail(8)

## Part 4: Combine and derive separations

Separations (departures) = accessions − net headcount change. If they hired 140 people and headcount fell by 50, then 190 people left.

In [ ]:
plot_df = pd.merge(emp_df[['date','headcount','net_change']], acc_df[['date','accessions']], on='date', how='inner')
plot_df['separations'] = plot_df['accessions'] - plot_df['net_change']
plot_df = plot_df[plot_df['date'] >= CHART_START].copy()
post_plot = post_df[post_df['date'] >= CHART_START].copy()
print(f"Chart period: {plot_df['date'].min().strftime('%b %Y')} → {plot_df['date'].max().strftime('%b %Y')}")

## Part 5: Chart

In [ ]:
C_HIRE = '#2E86AB'
C_SEP  = '#E84855'
C_TOT  = '#3D405B'
C_POST = '#6A4C93'
BG     = '#FAFAFA'
GRID   = '#E8E8E8'
FONT   = 'Helvetica Neue, Arial, sans-serif'

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['New Hires (Accessions)', 'Departures', 'Total Headcount', 'USAJobs Postings'],
    vertical_spacing=0.16,
    horizontal_spacing=0.10,
)

fig.add_trace(go.Bar(x=plot_df['date'], y=plot_df['accessions'],  marker_color=C_HIRE, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=plot_df['date'], y=plot_df['separations'], marker_color=C_SEP,  showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(
    x=plot_df['date'], y=plot_df['headcount'], mode='lines',
    line=dict(color=C_TOT, width=2.5),
    fill='tozeroy', fillcolor='rgba(61,64,91,0.08)',
    showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=post_plot['date'], y=post_plot['postings'], marker_color=C_POST, showlegend=False), row=2, col=2)

fig.add_vline(x=INAUGURATION, line=dict(color='#999', width=1.5, dash='dot'))

for ax_i in range(1, 5):
    xax = f'xaxis{ax_i}' if ax_i > 1 else 'xaxis'
    yax = f'yaxis{ax_i}' if ax_i > 1 else 'yaxis'
    fig.update_layout(**{
        xax: dict(showgrid=False, tickformat='%b %Y', dtick='M2', tickangle=-40,
                  tickfont=dict(size=9, family=FONT), zeroline=False),
        yax: dict(showgrid=True, gridcolor=GRID, zeroline=True, zerolinecolor='#ccc',
                  tickfont=dict(size=10, family=FONT)),
    })

fig.update_layout(
    yaxis3=dict(tickformat=',d'),
    title=dict(
        text='<b>DOJ: Hiring A Lot of Attorneys — But Still Shrinking</b><br>'
             '<span style="font-size:13px;color:#666">Series 0905 (General Attorney) · Jan 2024–Apr 2026</span>',
        font=dict(size=20, family=FONT, color='#1a1a2e'),
        x=0.5, xanchor='center',
    ),
    paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family=FONT, size=12, color='#333'),
    margin=dict(t=130, b=80, l=70, r=40),
    height=700, width=1150,
)

for ann in fig.layout.annotations:
    ann.font.update(size=13, family=FONT, color='#333')

fig.add_annotation(
    x=0.5, y=-0.09, xref='paper', yref='paper',
    text='Sources: OPM EHRI (impactproject/opm-ehri-data, HuggingFace) · USAJobs (abigailhaddad/usajobs_historical) · github.com/abigailhaddad/doj-lawyer-hiring',
    showarrow=False, font=dict(size=9, color='#aaa', family=FONT), xanchor='center',
)

fig.show()
fig.write_image('doj_4panel.png', scale=2)
print('Saved: doj_4panel.png')

## What the data shows

**Repo:** [github.com/abigailhaddad/doj-lawyer-hiring](https://github.com/abigailhaddad/doj-lawyer-hiring)

**The workforce is dramatically smaller.** DOJ peaked at ~13,130 series-0905 attorneys in early 2024. As of April 2026 (latest data), there are ~10,259 — a loss of ~2,871 attorneys (~22%). October 2025 saw a single-month drop of ~700, consistent with a RIF.

**The early-2025 hiring freeze was real.** New hires averaged ~27/month in Feb–Jun 2025 vs. ~75/month for the same period in 2024. Postings also collapsed to 12–15/month in January–February 2025.

**Postings have surged in 2026 — but mostly for U.S. Attorneys offices, not Main Justice divisions.** Monthly postings reached 121–165 in March–May 2026, compared to a 2024 baseline of 27–53/month. However, most of this increase comes from the Executive Office for U.S. Attorneys (AUSA district office positions), not from the litigating divisions analyzed in Part 6. Offices, Boards & Divisions went from 188 pre-inauguration postings to 144 post — a 23% drop.

**Accessions are back to normal levels, but headcount is still falling** because departures continue to outpace new hires.

## Division-level findings (Part 6, inferred — see note)

Divisions with politically sensitive enforcement mandates took the hardest hits; the criminal prosecution division was left largely untouched.

- **Civil Rights (−64%)**: The steepest real decline. Most of the drop came in a single quarter — July to October 2025 — consistent with a mass RIF event. Headcount went from 463 to 165 attorneys and has barely moved since.
- **ENRD (−38%)**: A steady grind down with no single cliff, just consistent monthly attrition.
- **Justice Management (−30%) / Antitrust (−24%)**: Gradual declines throughout.
- **Civil (−10%)**: Dipped to −14% by October 2025, then partially recovered — likely absorbing transfers from the Tax Division dissolution.
- **Criminal (~0%)**: Dipped to −16% by October 2025, then bounced back to roughly flat. The Tax Division was dissolved in December 2025 with all attorneys transferred out; many appear to have landed in Criminal.
- **Tax (−100%)**: Dissolved December 2025. Shows up as a cliff to zero, not a gradual decline.

## Notes

- EHRI data lags 1–3 months; the most recent months may be incomplete
- Some DOJ positions fill outside USAJobs (Schedule A, lateral transfers, honors program)
- Series 0905 spans entry-level to senior litigators across all DOJ divisions
- All EHRI data is anonymous aggregate counts — no individual-level information
- `separations` here is derived: `accessions − net_headcount_change`; it includes retirements, resignations, firings, and transfers out
- Division assignments in Part 6 are inferred from duty-station patterns, not confirmed by OPM or DOJ
- USAJobs postings are deduplicated by control number across historical and current parquet sources; see [`abigailhaddad/usajobs_historical`](https://github.com/abigailhaddad/usajobs_historical) README for details on the combining methodology

## Part 6: Headcount and Hiring by Main Justice Division

DOJ's "Offices, Boards and Divisions" (subelement **DJ01**) covers the traditional Main Justice litigating divisions. EHRI tracks them separately via a **Personnel Office Identifier (POI)** — a four-digit code OPM assigns to each distinct operating office.

We identify which division each POI represents using **duty-station fingerprints**: the set of cities where that POI's attorneys actually work. Each division has a recognizable geographic signature:

- **POI 1036** has a Denver cluster → Environment & Natural Resources Division (Rocky Mountain field office)
- **POI 1817** has a Dallas field office alongside DC → Tax Division (dissolved December 2025)
- **POI 1830** has SF/NY/Chicago offices → Antitrust Division (its known field presence)
- **POI 1818** is nearly all DC → Civil Rights Division (minimal field footprint)
- **POI 1034** shows Miami/LA/Houston/Chicago → Criminal Division

**Note:** POI-to-division assignments are inferred from duty-station patterns, not confirmed by OPM or DOJ. They are consistent with publicly known office locations but should be treated as estimates.

| POI  | Inferred Division | Key location signal |
|------|-------------------|---------------------|
| 1845 | Civil Division | Largest; scattered field offices |
| 1818 | Civil Rights Division | Nearly all DC |
| 1034 | Criminal Division | Miami / LA / Houston / Chicago |
| 1036 | Env & Natural Resources (ENRD) | Denver field office |
| 1830 | Antitrust Division | SF / NY / Chicago |
| 1831 | Justice Management Division | Suburban DC (Merrifield VA) |
| 1817 | Tax Division (dissolved Dec 2025) | Dallas field office |

In [ ]:
# POI → inferred division name (based on duty-station fingerprinting — see note above)
DIVISION_NAMES = {
    "1818": "Civil Rights",
    "1036": "ENRD",
    "1830": "Antitrust",
    "1034": "Criminal",
    "1845": "Civil",
    "1831": "Justice Mgmt",
    "1817": "Tax (dissolved)",
}
POI_SQL = ', '.join(f"'{p}'" for p in DIVISION_NAMES)

# Employment headcount by division — reuse emp_urls from Part 2
div_emp_df = conn.execute(f"""
    SELECT 
        regexp_extract(filename, 'employment_([0-9]{{6}})', 1) AS yyyymm,
        personnel_office_identifier_code AS poi,
        SUM(CAST(count AS INTEGER)) AS headcount
    FROM read_parquet([{emp_url_list}], filename=true)
    WHERE agency_subelement_code = 'DJ01'
      AND occupational_series_code = '0905'
      AND personnel_office_identifier_code IN ({POI_SQL})
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()
div_emp_df['date'] = pd.to_datetime(div_emp_df['yyyymm'], format='%Y%m')
div_emp_df['division'] = div_emp_df['poi'].map(DIVISION_NAMES)
div_emp_df = div_emp_df[div_emp_df['date'] >= CHART_START].copy()

# Accessions by division — reuse acc_urls from Part 1
div_acc_url_list = ', '.join(f"'{u}'" for u in acc_urls)
div_acc_df = conn.execute(f"""
    SELECT 
        regexp_extract(filename, 'accessions_([0-9]{{6}})', 1) AS yyyymm,
        personnel_office_identifier_code AS poi,
        SUM(CAST(count AS INTEGER)) AS accessions
    FROM read_parquet([{div_acc_url_list}], filename=true)
    WHERE agency_subelement_code = 'DJ01'
      AND occupational_series_code = '0905'
      AND personnel_office_identifier_code IN ({POI_SQL})
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()
div_acc_df['date'] = pd.to_datetime(div_acc_df['yyyymm'], format='%Y%m')
div_acc_df['division'] = div_acc_df['poi'].map(DIVISION_NAMES)
div_acc_df = div_acc_df[div_acc_df['date'] >= CHART_START].copy()

print("Employment:", div_emp_df['yyyymm'].min(), "→", div_emp_df['yyyymm'].max())
print("Accessions:", div_acc_df['yyyymm'].min(), "→", div_acc_df['yyyymm'].max())
div_emp_df.groupby('division')['headcount'].last().sort_values(ascending=False)

In [ ]:
DIVISIONS_ORDERED = [
    ("1845", "Civil"),
    ("1818", "Civil Rights"),
    ("1034", "Criminal"),
    ("1036", "ENRD"),
    ("1830", "Antitrust"),
    ("1831", "Justice Mgmt"),
    ("1817", "Tax (dissolved)"),
]
DIV_COLORS = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#ff7f0e', '#8c564b', '#7f7f7f']

NCOLS = 4
NROWS = 2
specs  = [[{"secondary_y": True}] * NCOLS for _ in range(NROWS)]
titles = [name for _, name in DIVISIONS_ORDERED] + [""]

fig_div = make_subplots(
    rows=NROWS, cols=NCOLS,
    specs=specs,
    subplot_titles=titles,
    vertical_spacing=0.22,
    horizontal_spacing=0.09,
)

for idx, (poi, name) in enumerate(DIVISIONS_ORDERED):
    row = idx // NCOLS + 1
    col = idx % NCOLS + 1
    color = DIV_COLORS[idx]

    emp = div_emp_df[div_emp_df['poi'] == poi].sort_values('date')
    acc = div_acc_df[div_acc_df['poi'] == poi].sort_values('date')

    fig_div.add_trace(
        go.Bar(x=acc['date'], y=acc['accessions'],
               marker_color=color, opacity=0.55,
               name=name, showlegend=False),
        row=row, col=col, secondary_y=False,
    )
    fig_div.add_trace(
        go.Scatter(x=emp['date'], y=emp['headcount'],
                   mode='lines', line=dict(color=color, width=2.5),
                   name=name, showlegend=False),
        row=row, col=col, secondary_y=True,
    )
    fig_div.add_vline(x=INAUGURATION, line=dict(color='#bbb', width=1, dash='dot'),
                      row=row, col=col)

fig_div.update_layout(
    title=dict(
        text=(
            '<b>DOJ Main Justice Divisions: Attorney Headcount & Monthly New Hires</b><br>'
            '<span style="font-size:11px;color:#666">'
            'Bars = monthly accessions (left axis) · Line = headcount (right axis) · '
            'Dotted line = Jan 20, 2025 · General Attorney series (0905) · DOJ Offices, Boards & Divisions'
            '</span>'
        ),
        font=dict(size=17, family=FONT, color='#1a1a2e'),
        x=0.5, xanchor='center',
    ),
    paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family=FONT, size=10, color='#333'),
    margin=dict(t=130, b=110, l=60, r=40),
    height=620, width=1150,
)
for ax in fig_div.select_yaxes():
    ax.update(showgrid=True, gridcolor=GRID, zeroline=False, tickfont=dict(size=8))
for ax in fig_div.select_xaxes():
    ax.update(showgrid=False, tickformat='%b %y', tickangle=-40, tickfont=dict(size=8))
for ann in fig_div.layout.annotations:
    ann.font.update(size=12, family=FONT, color='#333')

fig_div.add_annotation(
    x=0.5, y=-0.13, xref='paper', yref='paper', showarrow=False, xanchor='center',
    text='<b>Note:</b> Division labels are inferred from duty-station patterns in OPM data — not confirmed by OPM or DOJ. Treat as estimates.',
    font=dict(size=10, color='#c0392b', family=FONT),
)
fig_div.add_annotation(
    x=0.5, y=-0.18, xref='paper', yref='paper', showarrow=False, xanchor='center',
    text='Sources: OPM EHRI (impactproject/opm-ehri-data, HuggingFace) · github.com/abigailhaddad/doj-lawyer-hiring',
    font=dict(size=9, color='#aaa', family=FONT),
)

fig_div.show()
fig_div.write_image('doj_divisions.png', scale=2)
print('Saved: doj_divisions.png')

In [ ]:
# Index each division's headcount to Jan 2025 = 0%
baseline = div_emp_df[div_emp_df['yyyymm'] == '202501'].set_index('poi')['headcount']

indexed = div_emp_df.copy()
indexed['pct_change'] = indexed.apply(
    lambda r: (r['headcount'] / baseline.get(r['poi'], np.nan) - 1) * 100, axis=1
)

# Sort divisions by final pct value descending so legend order matches line order at right edge
final_pct = (
    indexed.sort_values('date')
    .groupby('poi')['pct_change']
    .last()
    .fillna(-100)
)
poi_color_map = dict(zip([poi for poi, _ in DIVISIONS_ORDERED], DIV_COLORS))
sorted_divisions = [(poi, name) for poi, name in
                    sorted(DIVISIONS_ORDERED, key=lambda x: final_pct.get(x[0], -100), reverse=True)]

fig_idx = go.Figure()
for poi, name in sorted_divisions:
    grp = indexed[indexed['poi'] == poi].sort_values('date')
    fig_idx.add_trace(go.Scatter(
        x=grp['date'], y=grp['pct_change'],
        mode='lines+markers',
        name=name,
        line=dict(color=poi_color_map[poi], width=2.5),
        marker=dict(size=5),
    ))

fig_idx.add_hline(y=0, line=dict(color='#aaa', width=1))
fig_idx.add_vline(x=INAUGURATION, line=dict(color='#999', width=1.5, dash='dot'),
                  annotation_text='Jan 20, 2025', annotation_position='top right',
                  annotation_font=dict(size=10, color='#999', family=FONT))

fig_idx.update_layout(
    title=dict(
        text=(
            '<b>DOJ Main Justice Divisions: Attorney Headcount Change Since Inauguration</b><br>'
            '<span style="font-size:12px;color:#666">'
            '% change from Jan 2025 · General Attorney series (0905) · DOJ Offices, Boards & Divisions'
            '</span>'
        ),
        font=dict(size=17, family=FONT, color='#1a1a2e'),
        x=0.5, xanchor='center',
    ),
    yaxis=dict(
        title='% change from Jan 2025',
        ticksuffix='%',
        showgrid=True, gridcolor=GRID,
        zeroline=True, zerolinecolor='#bbb', zerolinewidth=1.5,
        tickfont=dict(size=11, family=FONT),
    ),
    xaxis=dict(
        showgrid=False, tickformat='%b %Y', dtick='M2', tickangle=-30,
        tickfont=dict(size=10, family=FONT),
    ),
    legend=dict(
        orientation='v', x=1.01, y=1, xanchor='left',
        font=dict(size=11, family=FONT),
        bgcolor='rgba(0,0,0,0)',
    ),
    paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family=FONT, size=12, color='#333'),
    margin=dict(t=120, b=100, l=80, r=160),
    height=520, width=1100,
)

fig_idx.add_annotation(
    x=0.5, y=-0.17, xref='paper', yref='paper', showarrow=False, xanchor='center',
    text=(
        '<b>Note:</b> Division labels are inferred from duty-station patterns in OPM data — '
        'not confirmed by OPM or DOJ. Treat as estimates.'
    ),
    font=dict(size=10, color='#c0392b', family=FONT),
)
fig_idx.add_annotation(
    x=0.5, y=-0.23, xref='paper', yref='paper', showarrow=False, xanchor='center',
    text='Sources: OPM EHRI (impactproject/opm-ehri-data, HuggingFace) · github.com/abigailhaddad/doj-lawyer-hiring',
    font=dict(size=9, color='#aaa', family=FONT),
)

fig_idx.show()
fig_idx.write_image('doj_divisions_pct.png', scale=2)
print('Saved: doj_divisions_pct.png')